In [9]:
!git clone https://github.com/shiyu-coder/Kronos.git
%cd Kronos
!pip install -r /content/Kronos/requirements.txt yfinance

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 371 (delta 66), reused 49 (delta 49), pack-reused 269 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 23.58 MiB/s, done.
Resolving deltas: 100% (178/178), done.
/content/Kronos/Kronos


In [24]:
# NOTE: If running in Google Colab, execute these three lines in a separate cell first:
# !git clone https://github.com/shiyu-coder/Kronos.git
# %cd Kronos
# !pip install -r requirements.txt yfinance plotly

import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from model import Kronos, KronosTokenizer, KronosPredictor
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Define the Input DataFrame
portfolio_df = pd.DataFrame({
    'Ticker': ["NVDA", "CRWV", "MSFT", "RDDT", "NOW", "DRAM", "HIMS"]
})
portfolio_df['Lookback'] = portfolio_df['Ticker'].apply(lambda x: 400)
portfolio_df['Pred_Len'] = portfolio_df['Ticker'].apply(lambda x: 30)

# Ensure an output directory exists to save the plots
output_dir = "forecast_plots"
os.makedirs(output_dir, exist_ok=True)

df_list = []
x_timestamp_list = []
y_timestamp_list = []
valid_tickers = []

print("Fetching and formatting market data...")
for _, row in portfolio_df.iterrows():
    ticker = row['Ticker']
    lookback = row['Lookback']
    pred_len = row['Pred_Len']

    df = yf.Ticker(ticker).history(period="2y")
    if df.empty or len(df) < lookback:
        print(f"Skipping {ticker}: Not enough historical data.")
        continue

    df.reset_index(inplace=True)

    # Standardize columns for Kronos OHLCV expectations
    df.columns = df.columns.str.lower()
    df.rename(columns={'date': 'timestamps', 'datetime': 'timestamps'}, inplace=True)
    df['timestamps'] = pd.to_datetime(df['timestamps']).dt.tz_localize(None)

    # Calculate 50-day SMA for visual context
    df['sma_50'] = df['close'].rolling(window=50).mean()

    # Isolate context window to strictly enforce uniform batch length
    df_context = df.tail(lookback).reset_index(drop=True)

    x_df = df_context[['open', 'high', 'low', 'close', 'volume']]
    x_ts = df_context['timestamps']

    # Generate future business days
    last_date = x_ts.iloc[-1]
    y_ts = pd.Series(pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=pred_len))

    df_list.append(x_df)
    x_timestamp_list.append(x_ts)
    y_timestamp_list.append(y_ts)
    valid_tickers.append(ticker)

    # Store the historical context back in a dictionary so we can access the SMA for plotting later
    if not hasattr(portfolio_df, 'hist_data_cache'):
        portfolio_df.hist_data_cache = {}
    portfolio_df.hist_data_cache[ticker] = df_context

# 2. Load Model Architecture
print("\nLoading Kronos foundation model...")
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
predictor = KronosPredictor(model, tokenizer, max_context=512)

# 3. Generate Batch Predictions
print(f"\nRunning parallel batch prediction for {valid_tickers}...")
pred_df_list = predictor.predict_batch(
    df_list=df_list,
    x_timestamp_list=x_timestamp_list,
    y_timestamp_list=y_timestamp_list,
    pred_len=portfolio_df['Pred_Len'].iloc[0], # Assumes uniform prediction length
    T=1.0,
    top_p=0.9,
    sample_count=10, # Averages 10 paths for a stable trajectory
    verbose=True
)

# 4. Interactive Plotly Visualization and Saving
print("\nRendering and saving interactive charts...")

plot_window = 100 # Limit historical plot to 100 days

for ticker, pred_df in zip(valid_tickers, pred_df_list):
    hist_context = portfolio_df.hist_data_cache[ticker]
    hist_plot = hist_context.tail(plot_window).copy()

    # Combine timestamps into string categories to remove weekend gaps on the x-axis
    all_dates = pd.concat([hist_plot['timestamps'], pd.Series(pred_df.index)])
    date_strings = all_dates.dt.strftime('%Y-%m-%d').tolist()

    fig = go.Figure()

    # Trace 1: Historical Candlesticks
    fig.add_trace(go.Candlestick(
        x=date_strings[:len(hist_plot)],
        open=hist_plot['open'],
        high=hist_plot['high'],
        low=hist_plot['low'],
        close=hist_plot['close'],
        name='Historical Data'
    ))

    # Trace 2: 50-Day SMA
    fig.add_trace(go.Scatter(
        x=date_strings[:len(hist_plot)],
        y=hist_plot['sma_50'],
        mode='lines',
        line=dict(color='rgba(173, 216, 230, 0.8)', width=2),
        name='50-Day SMA'
    ))

    # Trace 3: Predicted Candlesticks
    fig.add_trace(go.Candlestick(
        x=date_strings[len(hist_plot):],
        open=pred_df['open'],
        high=pred_df['high'],
        low=pred_df['low'],
        close=pred_df['close'],
        name='Kronos Forecast',
        increasing_line_color='cyan',
        decreasing_line_color='magenta'
    ))

    fig.update_layout(
        title=f"{ticker} - {len(pred_df)}-Day Price Forecast (Kronos)",
        yaxis_title="Price (USD)",
        xaxis_title="Time (Days)",
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        hovermode="x unified",
        xaxis=dict(
            type='category',
            tickmode='auto',
            nticks=20
        )
    )

    # Display the plot in the Colab output cell
    fig.show()

    # Save the interactive plot as an HTML file locally
    file_path = os.path.join(output_dir, f"{ticker}_forecast.html")
    fig.write_html(file_path)
    print(f"Saved interactive plot to: {file_path}")

print(f"\nAll plots saved to the '{output_dir}' directory in Colab.")

Fetching and formatting market data...


/tmp/ipykernel_6600/768712460.py:67: UserWarning:

Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access



Skipping CRWV: Not enough historical data.
Skipping DRAM: Not enough historical data.

Loading Kronos foundation model...

Running parallel batch prediction for ['NVDA', 'MSFT', 'RDDT', 'NOW', 'HIMS']...


100%|██████████| 30/30 [23:41<00:00, 47.38s/it]



Rendering and saving interactive charts...


Saved interactive plot to: forecast_plots/NVDA_forecast.html


Saved interactive plot to: forecast_plots/MSFT_forecast.html


Saved interactive plot to: forecast_plots/RDDT_forecast.html


Saved interactive plot to: forecast_plots/NOW_forecast.html


Saved interactive plot to: forecast_plots/HIMS_forecast.html

All plots saved to the 'forecast_plots' directory in Colab.
